# Random Walk with Restart (RWR)

## Learning Objectives

1. **Distinguish** RWR from standard PageRank: single-source teleportation vs uniform
2. **Derive** the RWR power iteration: $r = (1-c)Mr + c\,e_Q$
3. **Interpret** RWR scores as a graph proximity / relevance measure
4. **Extend** to multi-seed RWR for broader query contexts
5. **Implement** RWR and apply it to recommendation and similarity search


## Problem Statement

### Personalized PageRank

Standard PageRank measures *global* importance. But often we want to know: **which nodes are most relevant to a specific query node $Q$?**

Examples:
- **Recommendation:** Given product $Q$, find the most similar products in a co-purchase graph
- **Biology:** Given disease gene $Q$, find related proteins in a protein-protein interaction network
- **Knowledge graphs:** Given entity $Q$, rank other entities by proximity

### Key Difference from PageRank

| | Standard PageRank | RWR |
|---|---|---|
| Teleport target | All nodes uniformly | Only the query node $Q$ |
| Score meaning | Global importance | Proximity to $Q$ |
| One vector per query | No (global) | Yes (one per seed) |

RWR assigns high score to nodes that a random walker starting at $Q$ — and restarting at $Q$ whenever it gets lost — tends to visit frequently.


In [ ]:
from IPython.display import SVG, display
svg = '''
<svg xmlns="http://www.w3.org/2000/svg" width="820" height="380" font-family="monospace" font-size="12">
  <rect width="820" height="380" fill="#fafafa" rx="8"/>
  <defs>
    <marker id="arr" markerWidth="9" markerHeight="7" refX="8" refY="3.5" orient="auto">
      <polygon points="0 0,9 3.5,0 7" fill="#666"/>
    </marker>
  </defs>

  <text x="410" y="22" text-anchor="middle" fill="#333" font-size="13" font-weight="bold">Random Walk with Restart (RWR) — Personalized PageRank</text>

  <!-- Graph -->
  <text x="185" y="46" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">Graph</text>

  <!-- query node Q (star) -->
  <circle cx="100" cy="110" r="28" fill="#f28e2b" stroke="#fff" stroke-width="2"/>
  <text x="100" y="106" text-anchor="middle" fill="white" font-size="12" font-weight="bold">Q</text>
  <text x="100" y="120" text-anchor="middle" fill="white" font-size="10">query</text>

  <!-- nodes -->
  <circle cx="240" cy="70"  r="22" fill="#4e79a7"/><text x="240" y="75"  text-anchor="middle" fill="white" font-size="12">A</text>
  <circle cx="240" cy="155" r="22" fill="#4e79a7"/><text x="240" y="160" text-anchor="middle" fill="white" font-size="12">B</text>
  <circle cx="340" cy="110" r="22" fill="#4e79a7"/><text x="340" y="115" text-anchor="middle" fill="white" font-size="12">C</text>
  <circle cx="340" cy="200" r="18" fill="#59a14f"/><text x="340" y="205" text-anchor="middle" fill="white" font-size="11">D</text>
  <circle cx="100" cy="220" r="18" fill="#76b7b2"/><text x="100" y="225" text-anchor="middle" fill="white" font-size="11">E</text>

  <!-- edges -->
  <line x1="128" y1="100" x2="218" y2="78"  stroke="#666" stroke-width="1.5" marker-end="url(#arr)"/>
  <line x1="128" y1="120" x2="218" y2="148" stroke="#666" stroke-width="1.5" marker-end="url(#arr)"/>
  <line x1="262" y1="78"  x2="318" y2="100" stroke="#666" stroke-width="1.5" marker-end="url(#arr)"/>
  <line x1="262" y1="155" x2="318" y2="118" stroke="#666" stroke-width="1.5" marker-end="url(#arr)"/>
  <line x1="340" y1="132" x2="340" y2="182" stroke="#666" stroke-width="1.5" marker-end="url(#arr)"/>
  <line x1="100" y1="138" x2="100" y2="202" stroke="#666" stroke-width="1.5" marker-end="url(#arr)"/>
  <line x1="318" y1="200" x2="122" y2="220" stroke="#666" stroke-width="1.5" marker-end="url(#arr)"/>

  <!-- RWR scores (bubble size proportional) -->
  <text x="185" y="270" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">RWR proximity scores from Q (c=0.15)</text>
  <circle cx="100" cy="330" r="30" fill="#f28e2b" opacity="0.7"/><text x="100" y="334" text-anchor="middle" fill="white" font-size="10">Q 0.42</text>
  <circle cx="190" cy="330" r="22" fill="#4e79a7" opacity="0.7"/><text x="190" y="334" text-anchor="middle" fill="white" font-size="10">A 0.21</text>
  <circle cx="265" cy="330" r="18" fill="#4e79a7" opacity="0.7"/><text x="265" y="334" text-anchor="middle" fill="white" font-size="10">B 0.18</text>
  <circle cx="330" cy="330" r="14" fill="#4e79a7" opacity="0.7"/><text x="330" y="334" text-anchor="middle" fill="white" font-size="10">C 0.11</text>
  <circle cx="385" cy="330" r="10" fill="#59a14f" opacity="0.7"/><text x="385" y="334" text-anchor="middle" fill="white" font-size="9">D 0.05</text>
  <circle cx="430" cy="330" r="8"  fill="#76b7b2" opacity="0.7"/><text x="430" y="334" text-anchor="middle" fill="white" font-size="9">E 0.03</text>

  <!-- RWR formula -->
  <rect x="460" y="46" width="345" height="150" rx="6" fill="#e8f0fb" stroke="#4e79a7" stroke-width="1.5"/>
  <text x="632" y="68" text-anchor="middle" fill="#4e79a7" font-size="12" font-weight="bold">RWR Update Rule</text>
  <text x="632" y="92" text-anchor="middle" fill="#333" font-size="13">r = (1−c) · M · r + c · e_Q</text>
  <text x="632" y="116" text-anchor="middle" fill="#555" font-size="11">c = restart probability (teleport back to Q)</text>
  <text x="632" y="136" text-anchor="middle" fill="#555" font-size="11">M = column-stochastic transition matrix</text>
  <text x="632" y="156" text-anchor="middle" fill="#555" font-size="11">e_Q = unit vector at query node Q</text>
  <text x="632" y="178" text-anchor="middle" fill="#555" font-size="10">(Personalized PageRank with single seed Q)</text>

  <!-- Vs PageRank -->
  <rect x="460" y="212" width="345" height="80" rx="6" fill="#fff4e0" stroke="#f28e2b" stroke-width="1.5"/>
  <text x="632" y="232" text-anchor="middle" fill="#f28e2b" font-size="11" font-weight="bold">RWR vs PageRank</text>
  <text x="632" y="252" text-anchor="middle" fill="#555" font-size="11">PageRank: teleport to ANY node uniformly</text>
  <text x="632" y="272" text-anchor="middle" fill="#555" font-size="11">RWR: teleport ONLY to query node Q</text>
  <text x="632" y="286" text-anchor="middle" fill="#555" font-size="10">→ score measures proximity to Q specifically</text>

  <!-- Applications -->
  <rect x="460" y="308" width="345" height="62" rx="6" fill="#e8f8e8" stroke="#59a14f" stroke-width="1.5"/>
  <text x="632" y="328" text-anchor="middle" fill="#59a14f" font-size="11" font-weight="bold">Applications</text>
  <text x="632" y="348" text-anchor="middle" fill="#555" font-size="11">• Recommendation: "similar to item Q"</text>
  <text x="632" y="362" text-anchor="middle" fill="#555" font-size="11">• Biology: protein-disease associations</text>
</svg>
'''
display(SVG(svg))


## Derivation

### Random Surfer with Restart

Define a random walk on the graph where at each step:
- With probability $1-c$: follow a uniformly random out-link
- With probability $c$: restart at the query node $Q$

This is exactly PageRank with the teleportation distribution concentrated on $Q$ instead of being uniform.

### Fixed-Point Equation

Let $r \in \mathbb{R}^n$ be the stationary distribution of this walk:

$$r = (1-c)\,M\,r + c\,e_Q$$

where $M$ is the column-stochastic transition matrix and $e_Q$ is the unit vector at $Q$.

**Comparison to PageRank:**  
PageRank: $r = \beta M r + (1-\beta)\frac{1}{n}\mathbf{1}$  
RWR: $r = (1-c)M r + c\,e_Q$

The only difference is the teleportation target: $\frac{1}{n}\mathbf{1}$ (uniform) vs $e_Q$ (single node).

### Power Iteration

Iterate until convergence:
$$r^{(t+1)} = (1-c)\,M\,r^{(t)} + c\,e_Q$$

Start from $r^{(0)} = e_Q$ (or any distribution). Convergence rate is $O((1-c)^t)$.

### Multi-Seed RWR

For multiple query nodes $Q_1, \ldots, Q_k$ with weights $w_1, \ldots, w_k$, replace $e_Q$ with the weighted mixture:
$$e = \sum_{j=1}^k w_j\, e_{Q_j}$$

This captures "similar to the *set* of query items" — used in content-based recommendation.


## Algorithm Steps

1. **Build** column-stochastic matrix $M$ from adjacency list
2. **Set** seed vector $e_Q$ (1 at query node, 0 elsewhere)
3. **Initialise** $r = e_Q$
4. **Iterate** $r \leftarrow (1-c)Mr + c\,e_Q$ until $\|r^{\text{new}} - r\|_1 < \epsilon$
5. **Rank** all nodes by their score in $r^*$


In [ ]:
import numpy as np


def random_walk_with_restart(adj, query_node, c=0.15, max_iter=100, tol=1e-8):
    """
    Random Walk with Restart (Personalized PageRank from a single source node).

    Inputs
    ------
    adj        : dict {node: list_of_neighbours}
    query_node : the seed node Q — walks restart here with probability c
    c          : float — restart probability (0.15 is typical)
    max_iter   : int
    tol        : float — L1 convergence tolerance

    Output
    ------
    scores : dict {node: rwr_score} — proximity to query_node; sums to 1
    """
    nodes = sorted(set(adj.keys()) | {v for nbrs in adj.values() for v in nbrs})
    n = len(nodes)
    idx = {node: i for i, node in enumerate(nodes)}

    # Column-stochastic transition matrix
    M = np.zeros((n, n))
    for src, nbrs in adj.items():
        if nbrs:
            prob = 1.0 / len(nbrs)
            for dst in nbrs:
                M[idx[dst], idx[src]] = prob
        else:
            M[:, idx[src]] = 1.0 / n  # dangling node

    # Seed vector e_Q
    e_q = np.zeros(n)
    e_q[idx[query_node]] = 1.0

    # Power iteration: r = (1-c)*M*r + c*e_Q
    r = e_q.copy()
    for _ in range(max_iter):
        r_new = (1 - c) * (M @ r) + c * e_q
        if np.sum(np.abs(r_new - r)) < tol:
            r = r_new
            break
        r = r_new

    return {nodes[i]: float(r[i]) for i in range(n)}


def rwr_multi_seed(adj, query_nodes, weights=None, c=0.15, max_iter=100, tol=1e-8):
    """
    RWR from multiple seed nodes (weighted combination).

    Inputs
    ------
    query_nodes : list of seed nodes
    weights     : list of floats (optional, uniform if None)
    """
    if weights is None:
        weights = [1.0 / len(query_nodes)] * len(query_nodes)
    else:
        s = sum(weights)
        weights = [w / s for w in weights]

    nodes = sorted(set(adj.keys()) | {v for nbrs in adj.values() for v in nbrs})
    n = len(nodes)
    idx = {node: i for i, node in enumerate(nodes)}

    M = np.zeros((n, n))
    for src, nbrs in adj.items():
        if nbrs:
            prob = 1.0 / len(nbrs)
            for dst in nbrs:
                M[idx[dst], idx[src]] = prob
        else:
            M[:, idx[src]] = 1.0 / n

    # Seed vector = weighted combination
    e_q = np.zeros(n)
    for node, w in zip(query_nodes, weights):
        e_q[idx[node]] += w

    r = e_q.copy()
    for _ in range(max_iter):
        r_new = (1 - c) * (M @ r) + c * e_q
        if np.sum(np.abs(r_new - r)) < tol:
            break
        r = r_new

    return {nodes[i]: float(r[i]) for i in range(n)}


# ── Demo ──────────────────────────────────────────────────────────────────────
adj = {
    "Q": ["A", "B"],
    "A": ["C"],
    "B": ["C"],
    "C": ["D"],
    "D": ["E"],
    "E": ["Q"],
}

scores = random_walk_with_restart(adj, query_node="Q", c=0.15)
print("RWR scores from Q (c=0.15):")
for node, score in sorted(scores.items(), key=lambda x: -x[1]):
    bar = "█" * int(score * 60)
    print(f"  {node}: {score:.4f}  {bar}")

# Show that changing query changes scores
print("\nRWR scores from C (c=0.15):")
scores_c = random_walk_with_restart(adj, query_node="C", c=0.15)
for node, score in sorted(scores_c.items(), key=lambda x: -x[1]):
    bar = "█" * int(score * 60)
    print(f"  {node}: {score:.4f}  {bar}")
